# Nuclear Receptor Ligand Analysis - All Families
## Replicating the Steroid Analysis Pipeline for All NR Families

### Analysis includes:
- **Step 2**: Molecular Property Analysis (descriptors + drug-likeness)
- **Step 3**: Chemical Space Analysis (correlation, histograms, PCA, t-SNE/UMAP, scaffolds)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, QED
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import FilterCatalog
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
import umap
import os
import warnings
warnings.filterwarnings('ignore')

# Plotting settings to match PDF style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ All imports successful!")

## Load Data

In [ ]:
# Load the complete dataset
input_file = "/mnt/user-data/uploads/ligands_with_final_chemical_classification.csv"
df_all = pd.read_csv(input_file)

print(f"✅ Total ligands: {len(df_all)}")
print(f"\n📊 NR Family distribution:")
print(df_all["NR_Family"].value_counts())

families = df_all["NR_Family"].unique()
print(f"\nFamilies to analyze: {list(families)}")

## Step 2: Molecular Property Calculation Function

In [ ]:
# Initialize PAINS filter (exactly as in the PDF)
params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog.FilterCatalog(params)

def calculate_molecular_properties(smiles):
    """Calculate molecular descriptors - matching steroid example from PDF"""
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return {col: None for col in [
            "MolWt", "TPSA", "NumHDonors", "NumHAcceptors", "LogP",
            "NumRotatableBonds", "QED", "RingCount", "AromaticRingCount",
            "StereoCenterCount", "PAINS_Flag", "Lipinski_Flag", "Veber_Flag"
        ]}
    
    # --- Basic & Physicochemical properties ---
    mol_wt = Descriptors.MolWt(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    logp = Crippen.MolLogP(mol)
    
    # --- Structural complexity ---
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    ring_count = Descriptors.RingCount(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    stereo_centers = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    
    # --- PAINS filtering ---
    pains_matches = pains_catalog.GetMatches(mol)
    pains_flag = "Fail" if pains_matches else "Pass"
    
    # --- Lipinski's Rule of Five ---
    lipinski_pass = (mol_wt <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)
    lipinski_flag = "Pass" if lipinski_pass else "Fail"
    
    # --- Veber's Rule ---
    veber_pass = (rot_bonds <= 10 and tpsa <= 140)
    veber_flag = "Pass" if veber_pass else "Fail"
    
    return {
        "MolWt": mol_wt,
        "TPSA": tpsa,
        "NumHDonors": hbd,
        "NumHAcceptors": hba,
        "LogP": logp,
        "NumRotatableBonds": rot_bonds,
        "QED": qed,
        "RingCount": ring_count,
        "AromaticRingCount": aromatic_rings,
        "StereoCenterCount": stereo_centers,
        "PAINS_Flag": pains_flag,
        "Lipinski_Flag": lipinski_flag,
        "Veber_Flag": veber_flag
    }

print("✅ Descriptor function defined")

## Process ALL Families - Calculate Descriptors

In [ ]:
# Create output directory
os.makedirs("family_analysis_results", exist_ok=True)

# Dictionary to store family dataframes
family_dfs = {}

print("="*80)
print("STEP 2: MOLECULAR PROPERTY ANALYSIS")
print("="*80 + "\n")

for family in families:
    print(f"Processing {family}...")
    
    # Filter for this family
    family_df = df_all[df_all["NR_Family"] == family].copy()
    print(f"  ✅ {len(family_df)} ligands found")
    
    # Calculate descriptors for each ligand
    descriptor_data = []
    for idx, row in family_df.iterrows():
        smiles = row.get("SMILES", "")
        props = calculate_molecular_properties(smiles)
        descriptor_data.append(props)
    
    # Add descriptors as new columns
    desc_df = pd.DataFrame(descriptor_data)
    family_df = pd.concat([family_df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)
    
    # Save to CSV
    family_clean = family.lower().replace(" ", "_").replace("-", "_")
    output_file = f"family_analysis_results/{family_clean}_with_full_descriptors.csv"
    family_df.to_csv(output_file, index=False)
    print(f"  💾 Saved: {output_file}")
    print(f"  📊 Columns: {len(family_df.columns)}\n")
    
    # Store for analysis
    family_dfs[family] = family_df

print("="*80)
print("✅ STEP 2 COMPLETE")
print("="*80)

## Step 3: Chemical Space Analysis - For Each Family
### Following the exact pipeline from the PDF steroid example

In [ ]:
def analyze_family_chemical_space(family_df, family_name):
    """
    Complete chemical space analysis matching PDF steroid example:
    - Figure 1: Correlation heatmap
    - Figure 2: Histograms (10 descriptors)
    - Figure 3: PCA plots (4 variations)
    - Figure 4: t-SNE and UMAP clustering
    - Figure 5: Scaffold analysis
    """
    
    family_clean = family_name.lower().replace(" ", "_").replace("-", "_")
    output_dir = f"family_analysis_results/{family_clean}_analysis"
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"ANALYZING: {family_name} Family")
    print(f"Total ligands: {len(family_df)}")
    print(f"{'='*80}\n")
    
    # === Select numeric descriptor columns ===
    descriptor_cols = [
        "MolWt", "TPSA", "NumHDonors", "NumHAcceptors", "LogP",
        "NumRotatableBonds", "QED", "RingCount", "AromaticRingCount",
        "StereoCenterCount"
    ]
    
    df_desc = family_df[descriptor_cols].copy()
    
    # ====================================================================
    # FIGURE 1: CORRELATION HEATMAP
    # ====================================================================
    print("📊 Creating Figure 1: Correlation Heatmap...")
    
    plt.figure(figsize=(12, 10))
    corr_matrix = df_desc.corr()
    
    sns.heatmap(corr_matrix, 
                annot=True, 
                fmt=".2f", 
                cmap="coolwarm", 
                center=0,
                square=True, 
                linewidths=1, 
                cbar_kws={"shrink": 0.8},
                vmin=-1, vmax=1)
    
    plt.title(f"Correlation Heatmap of {family_name} Descriptors", 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/figure1_correlation_heatmap.png", 
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: figure1_correlation_heatmap.png\n")
    
    # ====================================================================
    # FIGURE 2: HISTOGRAMS OF KEY DESCRIPTORS (10 plots)
    # ====================================================================
    print("📊 Creating Figure 2: Descriptor Histograms...")
    
    fig, axes = plt.subplots(3, 4, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, col in enumerate(descriptor_cols):
        if idx < len(axes):
            # Histogram
            axes[idx].hist(df_desc[col].dropna(), 
                          bins=25, 
                          color='skyblue', 
                          edgecolor='black', 
                          alpha=0.7)
            
            axes[idx].set_title(f"Distribution of {col}", fontweight='bold')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel("Count")
            axes[idx].grid(True, alpha=0.3)
            
            # Add KDE overlay
            try:
                ax2 = axes[idx].twinx()
                df_desc[col].dropna().plot(kind='kde', ax=ax2, color='red', linewidth=2)
                ax2.set_ylabel('')
                ax2.set_yticks([])
            except:
                pass
    
    # Hide unused subplots
    for idx in range(len(descriptor_cols), len(axes)):
        axes[idx].set_visible(False)
    
    fig.suptitle(f"{family_name} Ligand Descriptors - Distributions", 
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/figure2_histograms.png", 
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: figure2_histograms.png\n")
    
    # ====================================================================
    # FIGURE 3: PCA ANALYSIS (4 plots)
    # ====================================================================
    print("📊 Creating Figure 3: PCA Analysis...")
    
    # Drop rows with NaN
    df_clean = df_desc.dropna()
    family_df_clean = family_df.loc[df_clean.index].copy()
    
    print(f"   Rows with NaN dropped: {len(df_desc) - len(df_clean)}")
    print(f"   Clean dataset: {len(df_clean)} ligands")
    
    if len(df_clean) > 1:
        # Create PCA pipeline with standardization
        pca_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=2))
        ])
        
        # Fit and transform
        X_pca = pca_pipeline.fit_transform(df_clean)
        pca_model = pca_pipeline.named_steps['pca']
        
        # Compute combined drug-likeness score
        family_df_clean['Lipinski_Score'] = family_df_clean['Lipinski_Flag'].map({'Pass': 1, 'Fail': 0})
        family_df_clean['Veber_Score'] = family_df_clean['Veber_Flag'].map({'Pass': 1, 'Fail': 0})
        family_df_clean['DrugLikeness_Score'] = (
            family_df_clean['QED'] + 
            family_df_clean['Lipinski_Score'] + 
            family_df_clean['Veber_Score']
        ) / 3
        
        # Create 2x2 subplot for PCA
        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        
        var1 = pca_model.explained_variance_ratio_[0]
        var2 = pca_model.explained_variance_ratio_[1]
        
        # Plot 1: Colored by QED (continuous)
        scatter1 = axes[0,0].scatter(X_pca[:, 0], X_pca[:, 1], 
                                     c=family_df_clean['QED'], 
                                     cmap='viridis', 
                                     s=50, alpha=0.7, 
                                     edgecolors='black', linewidth=0.5)
        axes[0,0].set_xlabel(f'PCA1 ({var1:.1%} variance)', fontsize=11)
        axes[0,0].set_ylabel(f'PCA2 ({var2:.1%} variance)', fontsize=11)
        axes[0,0].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by QED', 
                           fontsize=12, fontweight='bold')
        axes[0,0].grid(True, alpha=0.3)
        cbar1 = plt.colorbar(scatter1, ax=axes[0,0])
        cbar1.set_label('QED', fontsize=10)
        
        # Plot 2: Colored by Lipinski Rule
        for flag, color in [('Pass', 'green'), ('Fail', 'red')]:
            mask = family_df_clean['Lipinski_Flag'] == flag
            axes[0,1].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                            c=color, label=f'Lipinski {flag}', 
                            s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
        axes[0,1].set_xlabel(f'PCA1 ({var1:.1%} variance)', fontsize=11)
        axes[0,1].set_ylabel(f'PCA2 ({var2:.1%} variance)', fontsize=11)
        axes[0,1].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by Lipinski Rule', 
                           fontsize=12, fontweight='bold')
        axes[0,1].legend(fontsize=10)
        axes[0,1].grid(True, alpha=0.3)
        
        # Plot 3: Colored by Veber Rule
        for flag, color in [('Pass', 'blue'), ('Fail', 'orange')]:
            mask = family_df_clean['Veber_Flag'] == flag
            axes[1,0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                            c=color, label=f'Veber {flag}', 
                            s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
        axes[1,0].set_xlabel(f'PCA1 ({var1:.1%} variance)', fontsize=11)
        axes[1,0].set_ylabel(f'PCA2 ({var2:.1%} variance)', fontsize=11)
        axes[1,0].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by Veber Rule', 
                           fontsize=12, fontweight='bold')
        axes[1,0].legend(fontsize=10)
        axes[1,0].grid(True, alpha=0.3)
        
        # Plot 4: Colored by Combined Drug-Likeness Score
        scatter4 = axes[1,1].scatter(X_pca[:, 0], X_pca[:, 1], 
                                     c=family_df_clean['DrugLikeness_Score'], 
                                     cmap='RdYlGn', 
                                     s=50, alpha=0.7, 
                                     edgecolors='black', linewidth=0.5,
                                     vmin=0, vmax=1)
        axes[1,1].set_xlabel(f'PCA1 ({var1:.1%} variance)', fontsize=11)
        axes[1,1].set_ylabel(f'PCA2 ({var2:.1%} variance)', fontsize=11)
        axes[1,1].set_title(f'PCA of {family_name} Ligand Chemical Space Colored by Combined Drug-Likeness Score', 
                           fontsize=12, fontweight='bold')
        axes[1,1].grid(True, alpha=0.3)
        cbar4 = plt.colorbar(scatter4, ax=axes[1,1])
        cbar4.set_label('Combined Drug-Likeness', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/figure3_pca_analysis.png", 
                    dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved: figure3_pca_analysis.png\n")
        
        # ====================================================================
        # FIGURE 4: t-SNE and UMAP CLUSTERING
        # ====================================================================
        print("📊 Creating Figure 4: t-SNE and UMAP Clustering...")
        
        # Standardize data
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(df_clean)
        
        # t-SNE
        perplexity = min(30, len(df_clean) - 1)
        tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
        X_tsne = tsne.fit_transform(X_scaled)
        
        # UMAP
        n_neighbors = min(15, len(df_clean) - 1)
        umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=n_neighbors)
        X_umap = umap_model.fit_transform(X_scaled)
        
        # KMeans clustering
        n_clusters = min(6, max(2, len(df_clean) // 50))
        kmeans_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('kmeans', KMeans(n_clusters=n_clusters, random_state=42, n_init=10))
        ])
        clusters = kmeans_pipeline.fit_predict(df_clean)
        
        # Create plots
        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        
        # t-SNE plot
        for cluster in range(n_clusters):
            mask = clusters == cluster
            axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                           label=f'Cluster {cluster}', 
                           s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
        axes[0].set_xlabel('tSNE1', fontsize=12)
        axes[0].set_ylabel('tSNE2', fontsize=12)
        axes[0].set_title(f't-SNE of {family_name} Ligands Colored by KMeans Cluster', 
                         fontsize=13, fontweight='bold')
        axes[0].legend(fontsize=9)
        axes[0].grid(True, alpha=0.3)
        
        # UMAP plot
        for cluster in range(n_clusters):
            mask = clusters == cluster
            axes[1].scatter(X_umap[mask, 0], X_umap[mask, 1], 
                           label=f'Cluster {cluster}', 
                           s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
        axes[1].set_xlabel('UMAP1', fontsize=12)
        axes[1].set_ylabel('UMAP2', fontsize=12)
        axes[1].set_title(f'UMAP of {family_name} Ligands Colored by KMeans Cluster', 
                         fontsize=13, fontweight='bold')
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/figure4_tsne_umap.png", 
                    dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ Saved: figure4_tsne_umap.png\n")
    
    # ====================================================================
    # FIGURE 5: SCAFFOLD ANALYSIS
    # ====================================================================
    print("📊 Creating Figure 5: Scaffold Analysis...")
    
    # Extract Murcko scaffolds
    def get_scaffold(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                scaffold = MurckoScaffold.GetScaffoldForMol(mol)
                return Chem.MolToSmiles(scaffold)
            return None
        except:
            return None
    
    family_df['Scaffold'] = family_df['SMILES'].apply(get_scaffold)
    scaffold_counts = family_df['Scaffold'].value_counts()
    
    # Statistics
    total_ligands = len(family_df)
    unique_scaffolds = scaffold_counts.nunique()
    scaffold_diversity = unique_scaffolds / total_ligands if total_ligands > 0 else 0
    
    print(f"   Total ligands: {total_ligands}")
    print(f"   🧬 Unique scaffolds: {unique_scaffolds}")
    print(f"   🔬 Scaffold diversity: {scaffold_diversity:.3f}")
    
    # Save scaffold statistics
    scaffold_stats = pd.DataFrame({
        'Scaffold': scaffold_counts.index,
        'Count': scaffold_counts.values
    })
    scaffold_stats.to_csv(f"{output_dir}/scaffold_statistics.csv", index=False)
    print(f"   💾 Saved: scaffold_statistics.csv")
    
    # Plot top 15 scaffolds
    top_n = min(15, len(scaffold_counts))
    top_scaffolds = scaffold_counts.head(top_n)
    
    plt.figure(figsize=(14, 8))
    bars = plt.barh(range(top_n), top_scaffolds.values, color='teal', edgecolor='black', linewidth=1.5)
    plt.yticks(range(top_n), top_scaffolds.index, fontsize=8, family='monospace')
    plt.xlabel('Ligand Count', fontsize=12, fontweight='bold')
    plt.ylabel('Scaffold (SMILES)', fontsize=12, fontweight='bold')
    plt.title(f'Top {top_n} Scaffolds in {family_name} Ligands', 
              fontsize=14, fontweight='bold', pad=15)
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    
    # Add count labels on bars
    for i, (bar, count) in enumerate(zip(bars, top_scaffolds.values)):
        plt.text(count + 0.5, i, str(count), 
                va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/figure5_scaffold_analysis.png", 
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: figure5_scaffold_analysis.png\n")
    
    print(f"{'='*80}")
    print(f"✅ COMPLETE: {family_name} analysis finished!")
    print(f"   All outputs in: {output_dir}/")
    print(f"{'='*80}\n")

print("✅ Analysis function ready")

## Run Complete Analysis for ALL Families

In [ ]:
print("\n" + "="*80)
print("STEP 3: CHEMICAL SPACE ANALYSIS - ALL FAMILIES")
print("="*80 + "\n")

# Analyze each family
for family in families:
    analyze_family_chemical_space(family_dfs[family], family)

print("\n" + "="*80)
print("🎉 ALL ANALYSES COMPLETE!")
print("="*80)

## Final Summary Report

In [ ]:
print("\n" + "="*80)
print("📊 COMPREHENSIVE SUMMARY")
print("="*80 + "\n")

summary_data = []

for family in families:
    df = family_dfs[family]
    family_clean = family.lower().replace(" ", "_").replace("-", "_")
    
    summary = {
        'Family': family,
        'N_Ligands': len(df),
        'Mean_MolWt': round(df['MolWt'].mean(), 2),
        'Mean_LogP': round(df['LogP'].mean(), 2),
        'Mean_TPSA': round(df['TPSA'].mean(), 2),
        'Mean_QED': round(df['QED'].mean(), 3),
        'Lipinski_Pass_%': round((df['Lipinski_Flag'] == 'Pass').mean() * 100, 1),
        'Veber_Pass_%': round((df['Veber_Flag'] == 'Pass').mean() * 100, 1),
        'PAINS_Pass_%': round((df['PAINS_Flag'] == 'Pass').mean() * 100, 1)
    }
    summary_data.append(summary)

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("family_analysis_results/COMPLETE_SUMMARY.csv", index=False)

print(summary_df.to_string(index=False))

print("\n" + "="*80)
print("📁 OUTPUT FILES GENERATED")
print("="*80)
print("""
family_analysis_results/
├── COMPLETE_SUMMARY.csv                      ← Summary statistics
│
├── steroid_with_full_descriptors.csv         ← Steroid descriptors
├── steroid_analysis/
│   ├── figure1_correlation_heatmap.png       ← Correlation matrix
│   ├── figure2_histograms.png                ← 10 descriptor distributions
│   ├── figure3_pca_analysis.png              ← 4 PCA plots
│   ├── figure4_tsne_umap.png                 ← Clustering analysis
│   ├── figure5_scaffold_analysis.png         ← Top 15 scaffolds
│   └── scaffold_statistics.csv               ← Scaffold data
│
├── non_steroid_with_full_descriptors.csv     ← Non-steroid descriptors
├── non_steroid_analysis/
│   └── [same 6 files as above]
│
├── orphan_with_full_descriptors.csv          ← Orphan descriptors  
└── orphan_analysis/
    └── [same 6 files as above]
""")

print("\n✅ Summary saved: family_analysis_results/COMPLETE_SUMMARY.csv")
print("\n🎉 ANALYSIS COMPLETE! All families analyzed with PDF-style visualizations.")
print("\n📊 Total outputs:")
print(f"   • {len(families)} descriptor CSV files")
print(f"   • {len(families) * 5} figure files")
print(f"   • {len(families)} scaffold statistics files")
print(f"   • 1 comprehensive summary file")
print("\n" + "="*80)